# MSFR — Low-Fidelity (0D DDE) vs High-Fidelity Comparison

This notebook builds the **zero-dimensional circulating-fuel MSFR model** (low fidelity, LF) and generates a multi-parameter dataset for an **unprotected loss of flow (ULOFF)** transient. The LF trajectories are then compared against **high-fidelity (HF) multi-physics OpenFOAM** simulations from the NuSHRED D2 dataset.

**Workflow**

1. Define reactor parameters and steady-state initial conditions for a 3000 MWth MSFR.
2. Assemble the circulating-fuel point-kinetics DDE and integrate ULOFF transients over a range of pump coast-down times $\tau_{\mathrm{pump}}$.
3. Persist the LF dataset to `dataset_lf_uloff.pkl`.
4. Reconstruct HF spatial fields from POD-compressed snapshots, spatially average over the in-core region, and overlay LF vs HF time histories.

**Dependencies:** `numpy`, `scipy`, `matplotlib`, `jitcdde`, `symengine`. HF comparison additionally uses the pre-computed dataset in `NuSHRED_Datasets/D2/` (see Paper 2 SVD pipeline).

## Setup

Import standard scientific Python libraries. NumPy and SciPy provide array operations and special functions; Matplotlib is used for plotting in the comparison section at the end of the notebook. The delay differential equation (DDE) solver (`jitcdde`) and symbolic engine (`symengine`) are imported later, when the model is assembled.

In [1]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

_Adapted from Cervi, E., Cammi, A., Di Ronco, A. Stability analysis of the Generation-IV nuclear reactors by means of the root
locus criterion (2018). Progress in Nuclear Energy 106, pp. 316-334, https://doi.org/10.1016/j.pnucene.2018.02.018_

The MSFR is a circulating-fuel nuclear reactor in which a mixture of
molten salts constitutes both the fuel and the coolant. In this work, a
3000 MWth MSFR is considered [1]; a thorium-fuelled molten salt mixture is employed as fuel [2]. The system features 16 coolant circuits, in which the
primary circuit and the power conversion system are separated by an
intermediate salt loop. FLiBe is adopted as intermediate coolant while
supercritical water at pressure of 22.5 MPa is chosen for the power
conversion system. The main design characteristics assumed in this work for the MSFR are taken from [1–2].

1. Merle-Lucotte, E., Heuer, D., Allibert, M., Brovchenko, M., Ghetta, V., Rubiolo, P., Laureau, A., 2013. Recommendations for a demonstrator of molten salt fast reactor. Proceedings of the IAEA International Conference on Fast Reactors and Related Fuel Cycles: Safe Technologies and Sustainable Scenarios (FR2013), Paris (France), 4–7 March 2013.
2. Ariu, V., 2014. Heat Exchanger Analysis for Innovative Molten Salt Fast Reactor. Master Thesis.

## Governing equations

The LF model couples **point kinetics with circulating delayed-neutron precursors**, a **lumped fuel energy balance**, and **linear fuel-temperature reactivity feedback**. The normalised power $\psi$ and precursor fractions $\eta_i$ in the reference formulation map to absolute power $P$ and precursor densities $C_i$ in the implementation below.

1. **Normalised point-kinetics** (one energy group, eight delayed-neutron precursor groups)
$$ \frac{d\psi(t)}{dt} = \frac{\rho(t)-\beta}{\Lambda} + \sum_i \frac{\beta_i}{\Lambda} \eta_i(t) $$
$$ \frac{d\eta_i(t)}{dt} = \lambda_i \psi(t) - \lambda_i \eta_i(t) + \frac{1}{\tau_f}\left(-\eta_i(t)+\eta_i(t-\tau_{ec})e^{-\lambda_i \tau_{ec}}\right)$$
2. **Fuel energy equation**
$$ m_f c_f \frac{dT_f}{dt} = \dot{Q}- 2 \Gamma_f c_f (T_{f}-T_{f,in}) $$
3. **Reactivity feedback**
$$ \rho(t) = \rho_{ext} + \alpha_f(T_f - T_f^0) $$

The precursor transport term introduces a **delay $\tau_{ec}$** equal to the external-loop transit time, which is the defining feature of circulating-fuel reactors and motivates the use of a DDE solver rather than a standard ODE integrator.

## Reactor parameters

Physical constants for a **3000 MWth MSFR** at the design point, taken from Merle-Lucotte et al. and Ariu (see references above).

- **Thermal hydraulics** — nominal power, fuel temperatures, salt mass flow rate $\Gamma_0$, core fuel inventory $M_f$, specific heat $c_f$, and the loop residence time $\tau_f = M_f/\Gamma_0$. The external heat-exchanger boundary temperature $T_{\mathrm{ext},0}$ is taken from OpenFOAM settings.
- **Neutronics** — eight-group delayed-neutron fractions $\beta_i$, decay constants $\lambda_i$, total delayed fraction $\beta$, precursor external-loop transit time $\tau_{ec}$, prompt neutron generation time $\Lambda$, and linear fuel temperature reactivity feedback $\alpha_f$.

Commented-out alternatives in the code block match OpenFOAM high-fidelity initial/boundary data used when comparing LF and HF results.


In [ ]:
# MSFR REACTOR DATA (Taken from [1-2] and OpenFOAM settings)

## ----------------------------------------------
## --------------- Thermal Hydraulics -----------
## ----------------------------------------------

P0 = 3000 * 10 ** 6         # [W] Core thermal power

TF0 = 700 + 273.15          # [K] Fuel average temperature
TFIn0 = 650 + 273.15        # [K] Core fuel inlet temperature
TFOut0 = 750 + 273.15       # [K] Core fuel outlet temperature

# TF0 = 1050.43               # [K] OpenFOAM Fuel temperature IC
# TFIn0 = 1000.43             # [K] OpenFOAM Fuel inlet temperature IC
# TFOut0 = 1100.43            # [K] OpenFOAM Fuel outlet temperature IC

Gamma0 = 18822              # [kg/s] Core fuel massa flow rate
MF = 37129                  # [kg] Total core fuel mass
CF = 1593.94                # [J/kgK] Fuel specific heat
tau_f = MF / Gamma0         # [s] Fuel characteristic time constant
T_ext0 = 900                # [K] HX external temperature (OpenFOAM)

## ----------------------------------------------
## --------------- Neutronics -------------------
## ----------------------------------------------

beta_i = np.array([
    24.8, 48.67, 41.54, 64.79, 95.48, 11.47, 19.22, 4.03
]) * 10 ** -5               # Tabulated values

# beta_i = np.array([
#     12.24, 75.7, 37.61, 81.37, 147.91, 51.99, 46.29, 15.74
# ]) * 10 ** -5             # OpenFOAM values

Beta = np.sum(beta_i)       # Delayed neutron fraction (sum)

tau_ec = 1.341              # [s] Characteristic transit time of precursors

Lambda = 1 * 10 ** -6       # [s] Neutron mean generation lifetime
lambda_i = np.array([
    0.0125, 0.0282, 0.0425, 0.133, 0.292, 0.666, 1.63, 3.55
])                          # [1/s] Delayed precursors decay constants

# alpha_f = -5 * 10 ** -5    # [1/K] Fuel feedback coefficient
alpha_f = -15 * 10 ** -5    # [1/K] Fuel feedback coefficient

## Steady-state initial conditions and transient input

At $t=0$ the core operates at nominal power $P_0$. Precursor concentrations $C_i(0)$ are computed from the circulating-fuel steady-state relation (Cervi et al.), including decay during the external loop transit $\tau_{ec}$:

$$C_i(0) = \frac{\beta_i P_0 / \Lambda}{\lambda_i + \bigl(1 - e^{-\lambda_i \tau_{ec}}\bigr)/\tau_f}.$$

The state vector is assembled as $Y_0 = [P_0, C_1, \ldots, C_8, T_f]$. Reactivity is chosen so the point-kinetics equations are balanced at startup:

$$\rho_0 = \beta - \sum_i \frac{\Lambda \lambda_i C_i(0)}{P_0}, \qquad \rho = \rho_{\mathrm{ext}} + \rho_0, \quad \rho_{\mathrm{ext}}=0.$$

**ULOFF input:** an unprotected loss of flow is modelled as exponential pump coast-down $u(t) = u_0\, e^{-t/\tau_{\mathrm{pump}}}$ with $u_0=1$. The default $\tau_{\mathrm{pump}}=5\,\mathrm{s}$ is used when building the model below; the parameter sweep later varies this time constant.

Commented-out assignments in the code block swap in **OpenFOAM-derived** precursor initial conditions when aligning LF and HF at the same physical ICs.

In [3]:
# Initial condition

C0 = np.divide(beta_i*P0/Lambda, lambda_i + (1-np.exp(-lambda_i*tau_ec))/tau_f)

# Initial condition from OpenFOAM

# C0 = np.array([
#     1.063e17, 3.77e17, 1.307e17, 9.586e16,
#     8.76e16, 3.079e16, 7.369e15, 1.235e15
# ])

Y0 = np.hstack(([P0], C0, [TF0]))

g = lambda IC: Y0

rho_ex = 0
rho_0 = Beta - np.sum(np.array([
    Lambda*lambda_i[i]*Y0[i+1]/P0 for i in range(8)
]))
rho = rho_ex + rho_0
# Input (ULOFF)

tau_pump = 5
u0 = 1

## Zero-dimensional MSFR model (low fidelity)

The `MSFR` function assembles the **circulating-fuel point-kinetics DDE** and returns the symbolic derivative list consumed by [`jitcdde`](https://jitcdde.rtfd.io/). State components:

| Index | Symbol | Description |
|-------|--------|-------------|
| `y(0)` | $P$ | Core thermal power |
| `y(1)`–`y(8)` | $C_i$ | Delayed-neutron precursor densities (8 groups) |
| `y(9)` | $T_f$ | Average core fuel temperature |

**Circulation scaling.** Pump speed $u(t)$ scales the fuel residence time $\tau_f$, precursor transit time $\tau_{ec}$, and mass flow $\Gamma$ inversely/proportionally.

**Neutronics.** Power follows one-group point kinetics with delayed precursors. Precursor equations include loop loss and return terms evaluated at the delayed state $y(\cdot, t-\tau_{ec})$, with decay $e^{-\lambda_i \tau_{ec}}$ over the external circuit.

**Thermal hydraulics.** A simplified heat-exchanger model increases effective cooling (`hx`) as the pump coasts down. Inlet temperature is blended between the delayed outlet estimate and $T_{\mathrm{ext},0}$. Fuel temperature evolves from a lumped energy balance with mixing factor `gamma = 15`.

This is the **LF surrogate** used to generate reference trajectories for comparison with HF OpenFOAM data.


In [4]:
from jitcdde import jitcdde, y, t #type:ignore
import symengine as sym #type:ignore

def MSFR(tau_pump):

    pump_coastdown = u0 * sym.exp(-t / tau_pump) if tau_pump is not None else 0

    t_f = tau_f / pump_coastdown if pump_coastdown != 0 else tau_f
    t_ec = tau_ec / pump_coastdown if pump_coastdown != 0 else tau_ec
    Gamma = Gamma0 * pump_coastdown if pump_coastdown != 0 else Gamma0

    p = y(0)
    c = [y(i) for i in range(1,9)]
    Tf = y(9)

    pd = y(0, t - tau_ec)
    cd = [y(i, t - tau_ec) for i in range(1,9)]
    Tfd = y(9, t - tau_ec)

    rho_ = rho + alpha_f * (Tf - Y0[-1])

    prec_sum = 0
    for i in range(8):
        prec_sum += lambda_i[i] * c[i]

    dPdt = (rho_-Beta)/Lambda*p + prec_sum

    dcdt = []
    for i in range(8):
        loss_rate =  c[i] / t_f
        return_rate = cd[i] / t_f * sym.exp(-lambda_i[i] * tau_ec)
        dcdt.append(
            beta_i[i]/Lambda*p - lambda_i[i]*c[i] - loss_rate + return_rate
        )

    # Simple first-order loop temperature tracking
    # As the pump slows down, heat exchanger residence time increases cooling efficiency
    hx = 0.5 + 0.5 * (1 - pump_coastdown)
    TFOutd = 2 * Tfd - T_ext0
    TFIn = TFOutd * (1.0 - hx) + T_ext0 * hx
    # TFIn = T_ext0
    ##

    # Artificial MF increase to account for mixing

    gamma = 15

    dTfdt = p/(gamma*MF*CF) - 2*Gamma/(gamma*MF) * (Tf - TFIn)

    f = [dPdt] + dcdt + [dTfdt]

    return pump_coastdown, f

pump_coastdown, f = MSFR(tau_pump)

## Multi-parameter transient study

The sections below generate a dataset over a range of ULOFF coast-down time constants. Each run recompiles and integrates the DDE independently so every trajectory corresponds to a different $\tau_{\mathrm{pump}}$.


### ULOFF — unprotected loss of flow

An **unprotected loss of flow** assumes the primary pumps coast down without scram or operator action. Circulation decay reduces heat removal and precursor advection, driving power and temperature transients.

The pump speed follows $u(t) = u_0 \exp(-t/\tau_{\mathrm{pump}})$ with $u_0 = 1$. We scan $\tau_{\mathrm{pump}}$ from **1 s to 10 s** (`np.linspace(1, 10, 21)` — 21 values). Faster coast-down (smaller $\tau$) removes circulation more abruptly and generally produces more severe transients.

For each $\tau_{\mathrm{pump}}$, the workflow is:

1. Instantiate the DDE with `MSFR(tau)` and JIT-compile the right-hand side to C via `jitcdde`.
2. Integrate from $t=0$ to $30\,\mathrm{s}$ on a uniform grid with $\Delta t = 10^{-4}\,\mathrm{s}$, starting from the constant past $Y_0$.
3. Record the pump input $u(t)$, the full state trajectory, the parameter value, and wall-clock integration time in `lf_times`.

Results are accumulated in `dataset_lf_uloff` with keys:

| Key | Content |
|-----|---------|
| `inputs` | Pump coast-down profile $u(t)$ for each $\tau$ |
| `outputs` | State trajectories; columns follow `var_names` |
| `param` | Coast-down time constant $\tau_{\mathrm{pump}}$ [s] |
| `var_names` | `q`, `prec1`–`prec8`, `T` (thermal power, eight precursor groups, fuel temperature) |
| `lf_times` | Wall-clock DDE integration time for each $\tau_{\mathrm{pump}}$ [s] |

> **Note:** `jitcdde` may warn about initial discontinuities on the first integration step; the code sets `initial_discontinuities_handled = True` after compilation. Recompilation per parameter value is intentional — the coast-down time enters the symbolic RHS.

In [ ]:
tau_uloff = np.linspace(1,10,21)
time_grid = np.arange(0, 30, 0.0001)

dataset_lf_uloff = {
    'inputs': [],
    'outputs': [],
    'param':[],
    'time_grid': time_grid,
    'var_names':['q','prec1','prec2','prec3','prec4','prec5','prec6', 'prec7', 'prec8', 'T']
}

lf_times = []

from time import time

for i in range(len(tau_uloff)):

    start_time = time()

    print(f'Running for index {i+1}/{len(tau_uloff)} (tau = {tau_uloff[i]})')

    pump_coast, f = MSFR(tau_uloff[i])

    DDE = jitcdde(f)
    DDE.compile_C()

    DDE.set_integration_parameters(
        atol = 1.0,
        rtol = 1e-5,
        min_step = 1e-13
    )
    DDE.initial_discontinuities_handled = True

    DDE.constant_past(Y0)

    results = []

    for _time in time_grid:
        state = DDE.integrate(_time)
        results.append(state)
        assert state.shape[0] == len(dataset_lf_uloff['var_names'])

    results = np.array(results)
    lf_times.append(time() - start_time)

    dataset_lf_uloff['param'].append(tau_uloff[i])
    dataset_lf_uloff['inputs'].append(u0 * np.exp(-time_grid / tau_uloff[i]))
    dataset_lf_uloff['outputs'].append(results)

dataset_lf_uloff['lf_times'] = lf_times


## Persist low-fidelity dataset

The parameter sweep produces 21 ULOFF trajectories (one per $\tau_{\mathrm{pump}}$). The dictionary `dataset_lf_uloff` is serialised to **`dataset_lf_uloff.pkl`** in the notebook directory.

This file is the LF reference used downstream by MF-SHRED training notebooks and by the HF comparison in the next section. Re-running the sweep overwrites the pickle; the `Results/` directory is created here for figure export.

In [6]:
import pickle as pkl
import os
path_data = '../../../NuSHRED_Datasets/D2/'

with open(path_data+'dataset_lf_uloff.pkl', 'wb') as f:
    pkl.dump(dataset_lf_uloff, f)

## Comparison with HF multi-physics (OpenFOAM) model

The high-fidelity reference comes from the **NuSHRED D2** dataset: three-dimensional, multi-physics OpenFOAM simulations of the same MSFR geometry under ULOFF boundary conditions. Those snapshots were compressed in the Paper 2 SVD pipeline (`Code/P2/01_svd.ipynb`) and stored under `NuSHRED_Datasets/D2/`.

**Goal:** assess whether the 0D lumped model captures the **in-core volume-averaged** transient response of key scalar fields — thermal power (`q`), delayed-neutron precursors (`prec1`–`prec8`), and fuel temperature (`T`) — across the same $\tau_{\mathrm{pump}}$ parameter range.

Because HF data are POD-compressed, fields are reconstructed from stored spatial modes before spatial averaging. LF and HF curves are normalised by their respective $t=0$ values so shapes can be compared on a common relative scale.

### Load HF compressed snapshots

The HF metadata bundle `msfr_p.uloff` provides parameter indices, field names, FOM time vector, per-field scalers, and POD mode counts. The computational mesh coordinates are in `domain.pkl`.

Together with the per-field POD bases (`pod_basis_<field>.svd`), singular values, and the joint temporal coefficient matrix `v_POD_all_fields.svd`, these artefacts allow reconstruction of any HF snapshot without reloading raw OpenFOAM fields.

In [7]:
domain = pkl.load(open(path_data+'domain.pkl', 'rb'))

import sys
sys.path.append('../../P2/')
idx_params, _, var_names, _, fom_times, scaler, Nmodes = pkl.load(open(path_data+'msfr_p.uloff', 'rb'))

### Overlay plots

For each LF field name in `dataset_lf_uloff['var_names']`:

1. Locate the matching entry in the HF `var_names` list.
2. Reconstruct the full spatial HF field from POD modes, singular values, and temporal coefficients (`v_POD`).
3. Apply `scaler.inverse_transform` to recover physical units.
4. Compute the **in-core spatial mean** using `in_core_idx` (nodes outside the hexagonal core envelope are excluded via `is_out_core`).
5. Plot **normalised** HF (solid red) and LF (dashed blue) traces on a $3 \times 7$ grid — one subplot per $\tau_{\mathrm{pump}}$ — and save to `Results/LF_Figs/<field>.pdf`.

Normalisation by the initial value highlights relative transient amplitude and timing differences without requiring identical absolute scaling between the reduced and full models.

In [ ]:
def is_out_core(node):
    return ((node[1] <= -1.0125 or node[1] >= 1.0125) or 
            node[0] >= 2.05)

out_core = np.array([is_out_core(node) for node in domain])
in_core_idx  = np.where(~out_core)[0]

v_pod = pkl.load(open(path_data+'CompressedDataset/v_POD_all_fields.svd', 'rb'))

path_results = 'Results/'
os.makedirs(path_results, exist_ok=True)

path_figs = path_results + 'LF_Figs/'
os.makedirs(path_figs, exist_ok=True)
for _idx_field in range(len(dataset_lf_uloff['var_names'])):
    field = dataset_lf_uloff['var_names'][_idx_field]

    assert field in var_names, f'{field} not in {var_names}'
    _idx = var_names.index(field)

    _pod_modes = pkl.load(open(path_data+'CompressedDataset/pod_basis_'+field+'.svd', 'rb'))
    _v_pod = v_pod[field]
    _sing_vals = pkl.load(open(path_data+'CompressedDataset/sing_vals_'+field+'.svd', 'rb'))

    fom_ave = np.zeros((len(dataset_lf_uloff['param']), len(fom_times)))

    for pp in range(len(dataset_lf_uloff['param'])):

        # If the full spatial field is desired with proper scaling
        _fom = scaler[_idx].inverse_transform(
            np.linalg.multi_dot([
                _pod_modes, _sing_vals, _v_pod[pp].T
            ])  
        )
        fom_ave[pp] = _fom[in_core_idx].mean(axis=0)

    nrows = 3
    ncols = len(dataset_lf_uloff['param']) // nrows

    fig, axs = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharex=True, sharey=True)
    axs = axs.ravel()

    for pp in range(len(dataset_lf_uloff['param'])):
        axs[pp].plot(fom_times, fom_ave[pp] / fom_ave[pp][0], 'r', label='OpenFOAM')
        axs[pp].plot(time_grid, dataset_lf_uloff['outputs'][pp][:, _idx_field] / dataset_lf_uloff['outputs'][pp][0, _idx_field], 'b--', label='ODE-Lumped')
        axs[pp].legend()
        axs[pp].set_title(f'{field} - '+r'$\tau_{{pump}}$'+f' = {dataset_lf_uloff["param"][pp]:.2f} s')

        if pp >= ncols * (nrows - 1):
            axs[pp].set_xlabel('Time (s)')
        
    plt.tight_layout()
    plt.savefig(path_figs + f'{field}.pdf', bbox_inches='tight', dpi = 200)
    plt.show()